# Description

This notebook provides the code to re-produce figures and tables for *The Lancet Countdown on Health and Climate Change* **Indicator 5.3.3: Political party engagement with health and climate change**

**Authors:** Zach Dickson & Cornelius Erfort 

In [1]:
#!pip install -q plotly pandas numpy nbformat kaleido scikit-learn matplotlib seaborn

## Primary Figure: Indicator 5.3.3: Political party engagement with health and climate change

In [2]:
# import libraries
import pandas as pd
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px


# read in data 
df = pd.read_csv('../data/indicator_5_3_3.csv')
# drop values after 2025 
df = df[df['year'] < 2025]

# dictionary to map party names to respective party colors
party_colors = {'Christian Democracy': '#1f77b4',
                'Conservative': '#9D755D',
                'Liberal': '#EECA3B',
                'Green': '#54A24B',
                'Radical Right-Wing': '#222A2A',
                'Social Democracy': '#E45756',}


# Define the mapping of data column -> subplot position
col_to_position = {
    'environment_climate_issue1_mean': (1, 1),  # Climate Change
    'healthcare_issue1_mean': (1, 2),           # Public Health
    'climate_health_mean': (2, 1),              # Climate and Health Nexus (mean)
    'climate_health_sum': (2, 2)                # Climate and Health Nexus (sum)
}


def create_nexus_subplot(df, 
                         title = 'Party Press Releases on Climate Change and Health by Party Family',
                         save_path=None,
                         col_to_position = col_to_position,
                         party_colors = party_colors):

    # Titles should match the order of the subplot grid
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Climate Change',
            'Public Health',
            'Climate & Health Nexus (mean)', 
            'Climate & Health Nexus (sum)'
        ),
        horizontal_spacing=0.05,  # reduce horizontal gap between columns
        vertical_spacing=0.15      # reduce vertical gap between rows
    )

    # Loop through the variables and plot each one in its correct subplot
    for i, col in enumerate(col_to_position.keys()):
        row, col_num = col_to_position[col]
        for party_family in party_colors.keys():
            subset = df[df['party_family'] == party_family].copy()

            # smooth the line
            subset[col] = subset[col].rolling(window=2, min_periods=1).mean()
            subset[col] = subset[col].rolling(window=2, min_periods=1).mean()

            fig.add_trace(go.Scatter(
                x=subset['year'],
                y=subset[col],
                mode='lines+markers',
                name=party_family,
                line=dict(color=party_colors[party_family], width=5),
                showlegend=True if i == 0 else False,
            ), row=row, col=col_num)

    # Update layout
    fig.update_layout(
        title=title,
        xaxis_title='Year',
        yaxis_title='Press Releases',
        title_x=0.05,
        legend_title='Party Family',
        hovermode='x unified',
        width=1000,
        height=700,
        template='presentation',
        font=dict(size=14),
        margin=dict(l=70, r=50, t=100, b=60)
    )

    # Format all y-axes as percentages
    fig.update_yaxes(tickformat=".0%", row=1, col=1)
    fig.update_yaxes(tickformat=".0%", row=1, col=2)
    fig.update_yaxes(tickformat=".0%", row=2, col=1)

    # Custom x-axis titles per subplot 
    fig.update_xaxes(title_text="Year", row=2, col=1)
    fig.update_xaxes(title_text="Year", row=2, col=2)
    fig.update_xaxes(title_text="Year", row=1, col=2)

    # add y-axis titles
    fig.update_yaxes(title_text="Press Releases", row=1, col=1)
    #fig.update_yaxes(title_text="Press Releases", row=1, col=2)
    fig.update_yaxes(title_text="Press Releases", row=2, col=1)
    #fig.update_yaxes(title_text="Press Releases", row=2, col=2)


    # save figure if save_path is provided
    if save_path:
        fig.write_image(save_path, scale=5)
    # show figure
    fig.show()


create_nexus_subplot(df, 
                     #title='',
                     save_path='../appendix/indicator_5_3_3.pdf',
                     col_to_position=col_to_position,
                     party_colors=party_colors)

In [3]:
# Generate individual subplot figures as separate PDFs

subplot_configs = {
    'environment_climate_issue1_mean': {
        'title': 'Climate Change',
        'filename': 'indicator_5_3_3_climate_change.pdf',
        'format_pct': True
    },
    'healthcare_issue1_mean': {
        'title': 'Public Health',
        'filename': 'indicator_5_3_3_public_health.pdf',
        'format_pct': True
    },
    'climate_health_mean': {
        'title': 'Climate & Health Nexus (mean)',
        'filename': 'indicator_5_3_3_nexus_mean.pdf',
        'format_pct': True
    },
    'climate_health_sum': {
        'title': 'Climate & Health Nexus (sum)',
        'filename': 'indicator_5_3_3_nexus_sum.pdf',
        'format_pct': False
    }
}

def create_single_subplot(df, col, config, save_path, party_colors=party_colors):
    """Create a single subplot figure and save it as PDF."""
    
    fig = go.Figure()
    
    for party_family in party_colors.keys():
        subset = df[df['party_family'] == party_family].copy()
        
        # smooth the line
        subset[col] = subset[col].rolling(window=2, min_periods=1).mean()
        subset[col] = subset[col].rolling(window=2, min_periods=1).mean()
        
        fig.add_trace(go.Scatter(
            x=subset['year'],
            y=subset[col],
            mode='lines+markers',
            name=party_family,
            line=dict(color=party_colors[party_family], width=5),
        ))
    
    # Update layout
    fig.update_layout(
        title=config['title'],
        xaxis_title='Year',
        yaxis_title='Press Releases',
        title_x=0.05,
        legend_title='Party Family',
        hovermode='x unified',
        width=1000,
        height=600,
        template='presentation',
        font=dict(size=14),
        margin=dict(l=70, r=50, t=80, b=60)
    )
    
    # Format y-axis as percentage if needed
    if config['format_pct']:
        fig.update_yaxes(tickformat=".0%")
    
    # Save figure
    fig.write_image(save_path, scale=5)
    print(f"Saved: {save_path}")

    # show figure
    fig.show()
    
# Generate each individual subplot
for col, config in subplot_configs.items():
    save_path = f"../appendix/{config['filename']}"
    create_single_subplot(df, col, config, save_path, party_colors)

Saved: ../appendix/indicator_5_3_3_climate_change.pdf


Saved: ../appendix/indicator_5_3_3_public_health.pdf


Saved: ../appendix/indicator_5_3_3_nexus_mean.pdf


Saved: ../appendix/indicator_5_3_3_nexus_sum.pdf


# Create table for appendix

In [4]:
# import libraries for classification report
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# read in validation data 
test = pd.read_parquet('../data/llm_validation_data.parquet')

# print classification report
print(classification_report(test['labels'].values, test.predicted_labels_integer.values))

# create confusion matrix
cm = confusion_matrix(test['labels'].values, test.predicted_labels_integer.values)

fig = px.imshow(cm,
                text_auto=True, 
                aspect="auto",
                color_continuous_scale='Blues',
                labels=dict(x="Predicted Label", y="True Label", color="Count"),
                title="Confusion Matrix",
)
fig.update_layout(
    height=600,
    width=600,
)
fig.show()

              precision    recall  f1-score   support

           0       0.96      0.94      0.95       308
           1       0.93      0.91      0.92        69
           2       0.97      0.97      0.97       195
           3       0.96      0.97      0.97        79
           4       0.90      0.95      0.93       132
           5       0.97      0.95      0.96       157
           6       0.97      0.94      0.95       168
           7       0.84      0.97      0.90        33
           8       0.95      0.94      0.95        88
           9       0.94      0.97      0.96        69
          10       0.95      0.95      0.95        79
          11       0.90      0.87      0.88        53
          12       0.94      0.96      0.95        48
          13       0.93      0.81      0.87        16
          14       0.84      0.97      0.90        80
          15       0.71      1.00      0.83         5
          16       0.92      0.69      0.79        16
          17       0.92    

## Country-level figures

Four figures showing attention to climate change, public health, and the climate–health nexus by country, 2010–2024. Countries are coloured by EEA sub-region (Western / Eastern / Northern / Southern / UK). Each panel uses an independent y-axis to show within-country trends clearly.

In [5]:
# ── Country-level figures ─────────────────────────────────────────────────────

df_country = pd.read_csv('../data/indicator_5_3_3_country.csv')
df_country = df_country[df_country['year'] < 2025]

# ── EEA sub-region mapping ───────────────────────────────────────────────────
country_region = {
    'Austria': 'Western', 'Belgium': 'Western', 'France': 'Western',
    'Germany': 'Western', 'Netherlands': 'Western', 'Switzerland': 'Western',
    'Bulgaria': 'Eastern', 'Czech Republic': 'Eastern', 'Hungary': 'Eastern',
    'Poland': 'Eastern', 'Romania': 'Eastern', 'Slovakia': 'Eastern',
    'Denmark': 'Northern', 'Estonia': 'Northern', 'Finland': 'Northern',
    'Iceland': 'Northern', 'Ireland': 'Northern', 'Lithuania': 'Northern',
    'Norway': 'Northern', 'Sweden': 'Northern',
    'Croatia': 'Southern', 'Cyprus': 'Southern', 'Greece': 'Southern',
    'Italy': 'Southern', 'Portugal': 'Southern', 'Slovenia': 'Southern',
    'Spain': 'Southern', 'UK': 'Not EEA',
}
df_country['EEA sub-region division'] = df_country['country'].map(country_region).fillna('Not EEA')

# ── Infer total press releases per country-year from sum/mean ────────────────
def infer_n_total(row):
    for sum_col, mean_col in [('environment_climate_issue1_sum', 'environment_climate_issue1_mean'),
                               ('healthcare_issue1_sum',          'healthcare_issue1_mean'),
                               ('climate_health_sum',             'climate_health_mean')]:
        if row[mean_col] > 0:
            return round(row[sum_col] / row[mean_col])
    return np.nan

df_country['n_total'] = df_country.apply(infer_n_total, axis=1)

MIN_N = 200

# ── EEA sub-region colours ───────────────────────────────────────────────────
region_colors = {
    'Western':  '#1f77b4',
    'Eastern':  '#d62728',
    'Northern': '#2ca02c',
    'Southern': '#ff7f0e',
    'Not EEA':  '#7f7f7f',
}

country_subplot_configs = {
    'environment_climate_issue1_mean': {
        'title': 'Climate Change by Country',
        'filename': 'indicator_5_3_3_country_climate_change.pdf',
        'format_pct': True,
    },
    'healthcare_issue1_mean': {
        'title': 'Public Health by Country',
        'filename': 'indicator_5_3_3_country_public_health.pdf',
        'format_pct': True,
    },
    'climate_health_mean': {
        'title': 'Climate & Health Nexus (mean) by Country',
        'filename': 'indicator_5_3_3_country_nexus_mean.pdf',
        'format_pct': True,
    },
    'climate_health_sum': {
        'title': 'Climate & Health Nexus (sum) by Country',
        'filename': 'indicator_5_3_3_country_nexus_sum.pdf',
        'format_pct': False,
    },
}

countries_sorted = sorted(df_country['country'].unique())
df_country['country'] = pd.Categorical(df_country['country'],
                                        categories=countries_sorted, ordered=True)
NCOLS = 6


def create_country_figure(df, col, config, save_path, min_n=MIN_N):
    """
    Faceted small-multiples figure: one panel per country, coloured by EEA sub-region.

    Data quality: years with fewer than `min_n` total press releases are shown
    as dashed grey; years above the threshold are solid coloured. None values are
    inserted so that the two traces never overlap and lines break cleanly at
    transitions between reliable and unreliable stretches.
    Panel titles show each country's median annual press-release count.
    """
    smoothed = []
    for country in countries_sorted:
        sub = df[df['country'] == country].copy().sort_values('year')
        sub[col] = sub[col].rolling(window=2, min_periods=1).mean()
        sub[col] = sub[col].rolling(window=2, min_periods=1).mean()
        smoothed.append(sub)
    df_sm = pd.concat(smoothed, ignore_index=True)

    nrows = -(-len(countries_sorted) // NCOLS)

    median_n = df.groupby('country')['n_total'].median().reindex(countries_sorted)
    panel_titles = [
        f"{c}<br><sup>median n={int(median_n[c]):,}</sup>"
        if pd.notna(median_n[c]) else f"{c}<br><sup>median n=?</sup>"
        for c in countries_sorted
    ]

    fig = make_subplots(
        rows=nrows, cols=NCOLS,
        subplot_titles=panel_titles,
        horizontal_spacing=0.06,
        vertical_spacing=0.15,
    )

    seen_regions = set()
    lo_legend_added = False

    for idx, country in enumerate(countries_sorted):
        row     = idx // NCOLS + 1
        col_num = idx % NCOLS + 1
        sub     = df_sm[df_sm['country'] == country].sort_values('year')

        region_label = df[df['country'] == country]['EEA sub-region division'].iloc[0]
        color = region_colors.get(region_label, '#888888')

        x_hi, y_hi = [], []
        x_lo, y_lo = [], []
        for _, r in sub.iterrows():
            is_reliable = pd.notna(r['n_total']) and r['n_total'] >= min_n
            if is_reliable:
                x_hi.append(r['year']);  y_hi.append(r[col])
                x_lo.append(None);       y_lo.append(None)
            else:
                x_hi.append(None);       y_hi.append(None)
                x_lo.append(r['year']);  y_lo.append(r[col])

        fig.add_trace(
            go.Scatter(
                x=x_hi, y=y_hi,
                mode='lines+markers',
                name=region_label,
                line=dict(color=color, width=2),
                marker=dict(size=4),
                connectgaps=False,
                showlegend=(region_label not in seen_regions),
                legendgroup=region_label,
            ),
            row=row, col=col_num,
        )
        seen_regions.add(region_label)

        if any(v is not None for v in y_lo):
            fig.add_trace(
                go.Scatter(
                    x=x_lo, y=y_lo,
                    mode='lines+markers',
                    name=f'< {min_n} press releases',
                    line=dict(color='#aaaaaa', width=1.5, dash='dash'),
                    marker=dict(size=3, color='#aaaaaa'),
                    connectgaps=False,
                    showlegend=not lo_legend_added,
                    legendgroup='low_n',
                ),
                row=row, col=col_num,
            )
            lo_legend_added = True

        if config['format_pct']:
            fig.update_yaxes(tickformat='.0%', row=row, col=col_num)

    fig.update_layout(
        title=dict(
            text=(f"{config['title']}"
                  f"<br><sup>Dashed grey = fewer than {min_n:,} press releases "
                  f"that year (estimates unreliable)</sup>"),
            x=0.05,
        ),
        width=1400,
        height=220 * nrows + 120,
        template='presentation',
        font=dict(size=11),
        legend_title='EEA Sub-region',
        margin=dict(l=50, r=30, t=100, b=50),
    )

    fig.write_image(save_path, scale=3)
    print(f'Saved: {save_path}')
    fig.show()


for col, config in country_subplot_configs.items():
    save_path = f"../appendix/{config['filename']}"
    create_country_figure(df_country, col, config, save_path)

/tmp/ipykernel_788849/2893274656.py:92: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Saved: ../appendix/indicator_5_3_3_country_climate_change.pdf


/tmp/ipykernel_788849/2893274656.py:92: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Saved: ../appendix/indicator_5_3_3_country_public_health.pdf


/tmp/ipykernel_788849/2893274656.py:92: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Saved: ../appendix/indicator_5_3_3_country_nexus_mean.pdf


/tmp/ipykernel_788849/2893274656.py:92: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



Saved: ../appendix/indicator_5_3_3_country_nexus_sum.pdf


## Party-level figures

One figure per country (Poland, Sweden, Germany, Spain, Portugal, France, Italy), with one subplot per party. Parties with fewer than 100 total press releases are excluded. Years with fewer than 30 press releases are shown as dashed grey (low-reliability).

In [6]:
# ── Party-level figures ────────────────────────────────────────────────────────

import polars as pl
import pandas as pd

TARGET_COUNTRIES  = ['France', 'Germany', 'Italy', 'Poland', 'Portugal', 'Spain', 'Sweden']
MIN_PARTY_TOTAL   = 100
MIN_PARTY_YEAR    = 30
NCOLS_PARTY       = 3

PARTY_COLORS = {
    'climate':  '#2ca02c',
    'health':   '#d62728',
    'nexus':    '#1f77b4',
}

# ── Load parlgov for party_family ─────────────────────────────────────────────
_parlgov = (
    pd.read_excel('../../../parlgov_parties.xlsx', usecols=['party_id', 'family_name'])
    .rename(columns={'family_name': 'party_family'})
    .assign(party_id=lambda x: pd.to_numeric(x['party_id'], errors='coerce'))
    .dropna(subset=['party_id'])
    .astype({'party_id': int})
)
parlgov = pl.from_pandas(_parlgov)

# ── Load & aggregate from classified parquet ──────────────────────────────────
_raw = pl.read_parquet('../../../press_releases_all_with_CAP_issues.parquet')

# date column is already datetime — extract year directly
_raw = (
    _raw
    .with_columns(
        pl.col('date').dt.year().alias('year')
    )
    .filter(pl.col('year') < 2025)
)

# Derive binary issue columns
_raw = _raw.with_columns([
    (pl.col('CAP_issue1') == 'environment and climate').cast(pl.Int8).alias('environment_climate_issue1'),
    (pl.col('CAP_issue1') == 'healthcare').cast(pl.Int8).alias('healthcare_issue1'),
    (
        ((pl.col('CAP_issue1') == 'environment and climate') & (pl.col('CAP_issue2') == 'healthcare')) |
        ((pl.col('CAP_issue1') == 'healthcare') & (pl.col('CAP_issue2') == 'environment and climate'))
    ).cast(pl.Int8).alias('climate_health'),
])

# Join with parlgov to attach party_family
_raw = (
    _raw
    .with_columns(pl.col('parlgov_party_id').cast(pl.Int64, strict=False))
    .join(
        parlgov.with_columns(pl.col('party_id').cast(pl.Int64)),
        left_on='parlgov_party_id',
        right_on='party_id',
        how='left',
    )
)

df_party = (
    _raw
    .group_by(['country', 'party', 'party_family', 'year'])
    .agg([
        pl.col('environment_climate_issue1').mean().alias('environment_climate_mean'),
        pl.col('healthcare_issue1').mean().alias('healthcare_mean'),
        pl.col('climate_health').mean().alias('climate_health_mean'),
        pl.col('environment_climate_issue1').count().alias('n_total'),
    ])
    .sort(['country', 'party', 'year'])
).to_pandas()

# Exclude parties with too few total press releases
party_totals = df_party.groupby(['country', 'party'])['n_total'].sum()
valid_parties = party_totals[party_totals >= MIN_PARTY_TOTAL].reset_index()[['country', 'party']]
df_party = df_party.merge(valid_parties, on=['country', 'party'], how='inner')

print(f"Rows after filtering: {len(df_party):,}")
print(df_party.groupby('country')['party'].nunique().reindex(TARGET_COUNTRIES))


# ── create_party_figure ───────────────────────────────────────────────────────

def create_party_figure(df, country, save_dir, min_year_n=MIN_PARTY_YEAR):
    """
    One subplot per party for a given country.
    Each subplot shows 3 traces: Climate (green), Health (red), Nexus (blue).
    Years with fewer than min_year_n press releases → dashed grey per metric.
    """
    sub_country = df[df['country'] == country].copy()
    parties = sorted(sub_country['party'].unique())

    nrows = -(-len(parties) // NCOLS_PARTY)
    metric_cols = ['environment_climate_mean', 'healthcare_mean', 'climate_health_mean']
    metric_labels = ['Climate Change', 'Public Health', 'Nexus']
    metric_colors = [PARTY_COLORS['climate'], PARTY_COLORS['health'], PARTY_COLORS['nexus']]

    smoothed_frames = []
    for party in parties:
        sub = sub_country[sub_country['party'] == party].copy().sort_values('year')
        for col in metric_cols:
            sub[col] = sub[col].rolling(window=2, min_periods=1).mean()
            sub[col] = sub[col].rolling(window=2, min_periods=1).mean()
        smoothed_frames.append(sub)
    df_sm = pd.concat(smoothed_frames, ignore_index=True)

    def panel_title(party):
        rows = sub_country[sub_country['party'] == party]
        family = rows['party_family'].iloc[0] if len(rows) else ''
        med_n = rows['n_total'].median()
        n_str = f"{int(med_n):,}" if pd.notna(med_n) else '?'
        return f"<b>{party}</b><br><sup>{family} · median n/yr={n_str}</sup>"

    panel_titles = [panel_title(p) for p in parties]

    fig = make_subplots(
        rows=nrows, cols=NCOLS_PARTY,
        subplot_titles=panel_titles,
        horizontal_spacing=0.08,
        vertical_spacing=0.18,
    )

    legend_added = {label: False for label in metric_labels}
    lo_legend_added = False

    for pidx, party in enumerate(parties):
        row     = pidx // NCOLS_PARTY + 1
        col_num = pidx % NCOLS_PARTY + 1
        sub     = df_sm[df_sm['party'] == party].sort_values('year')
        sub_raw = sub_country[sub_country['party'] == party].sort_values('year')

        for col, label, color in zip(metric_cols, metric_labels, metric_colors):
            x_hi, y_hi = [], []
            x_lo, y_lo = [], []

            for _, r in sub.iterrows():
                raw_n = sub_raw.loc[sub_raw['year'] == r['year'], 'n_total']
                n = raw_n.iloc[0] if len(raw_n) else np.nan
                is_reliable = pd.notna(n) and n >= min_year_n

                if is_reliable:
                    x_hi.append(r['year']); y_hi.append(r[col])
                    x_lo.append(None);      y_lo.append(None)
                else:
                    x_hi.append(None);      y_hi.append(None)
                    x_lo.append(r['year']); y_lo.append(r[col])

            fig.add_trace(
                go.Scatter(
                    x=x_hi, y=y_hi,
                    mode='lines+markers',
                    name=label,
                    line=dict(color=color, width=2),
                    marker=dict(size=4),
                    connectgaps=False,
                    showlegend=not legend_added[label],
                    legendgroup=label,
                ),
                row=row, col=col_num,
            )
            legend_added[label] = True

            if any(v is not None for v in y_lo):
                fig.add_trace(
                    go.Scatter(
                        x=x_lo, y=y_lo,
                        mode='lines+markers',
                        name=f'< {min_year_n} press releases',
                        line=dict(color='#aaaaaa', width=1.5, dash='dash'),
                        marker=dict(size=3, color='#aaaaaa'),
                        connectgaps=False,
                        showlegend=not lo_legend_added,
                        legendgroup='low_n',
                    ),
                    row=row, col=col_num,
                )
                lo_legend_added = True

        fig.update_yaxes(tickformat='.0%', row=row, col=col_num)

    fig.update_layout(
        title=dict(
            text=(f"<b>{country}</b> — Party-level attention to Climate, Health & Nexus"
                  f"<br><sup>Dashed grey = fewer than {min_year_n} press releases "
                  f"that year (estimates unreliable)</sup>"),
            x=0.05,
        ),
        width=1400,
        height=280 * nrows + 140,
        template='presentation',
        font=dict(size=11),
        legend_title='Issue',
        margin=dict(l=50, r=30, t=120, b=50),
    )

    slug = country.lower().replace(' ', '_')
    save_path = f"{save_dir}/indicator_5_3_3_party_{slug}.pdf"
    fig.write_image(save_path, scale=3)
    print(f"Saved: {save_path}")
    fig.show()


# ── Generate one figure per target country ───────────────────────────────────
import os
save_dir = '../appendix'
os.makedirs(save_dir, exist_ok=True)

for country in TARGET_COUNTRIES:
    if country in df_party['country'].values:
        create_party_figure(df_party, country, save_dir)
    else:
        print(f"WARNING: '{country}' not found in data — skipping.")

Rows after filtering: 1,300
country
France      1
Germany     7
Italy       4
Poland      8
Portugal    1
Spain       5
Sweden      9
Name: party, dtype: int64
Saved: ../appendix/indicator_5_3_3_party_france.pdf


Saved: ../appendix/indicator_5_3_3_party_germany.pdf


Saved: ../appendix/indicator_5_3_3_party_italy.pdf


Saved: ../appendix/indicator_5_3_3_party_poland.pdf


Saved: ../appendix/indicator_5_3_3_party_portugal.pdf


Saved: ../appendix/indicator_5_3_3_party_spain.pdf


Saved: ../appendix/indicator_5_3_3_party_sweden.pdf
